# PINN State Estimation — Final Version v3
## Target: Systems & Control Letters (Elsevier)

### Key fixes in this version:
- **Fix A:** Torques reduced 10x — angles stay below 0.3 rad, no tumbling
- **Fix B:** Altitude feedback on thrust — z stays bounded near 5m throughout
- **Fix C:** 5 second simulation — prevents long-horizon drift
- **Fix D:** normalize_obs() used everywhere — no broadcast errors
- **Fix E:** All normalization consistent throughout all cells

### Kaggle Instructions
1. **Accelerator:** Settings → Accelerator → **GPU T4 x2** (or P100)
2. **Run all:** Run → Run All
3. All figures are saved to **/kaggle/working/** and appear in the Output panel on the right
4. Download figures individually from the Output panel, or use the dataset export button


In [ ]:
# CELL 1: Install missing packages (torch, numpy, matplotlib pre-installed on Kaggle)
!pip install scipy scikit-learn tqdm -q


In [ ]:
# CELL 2: Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'axes.titlesize': 13,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 1.8,
})
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Kaggle output directory — all figures will be saved here
import os
OUT_DIR = '/kaggle/working/'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')

---
## Figure 1 — Van der Pol Oscillator

In [ ]:
# CELL 3: Van der Pol simulation
mu_vdp = 2.0
t_end_vdp = 20.0
t_vdp  = np.arange(0, t_end_vdp, 0.05)
N_vdp  = len(t_vdp)
x0_vdp = [2.0, 0.0]

def vdp(t, state):
    x1, x2 = state
    return [x2, mu_vdp*(1 - x1**2)*x2 - x1]

sol_vdp    = solve_ivp(vdp, (0, t_end_vdp), x0_vdp, t_eval=t_vdp, method='RK45', rtol=1e-8, atol=1e-10)
states_vdp = sol_vdp.y.T

obs_idx_vdp = np.sort(np.random.choice(N_vdp, int(N_vdp*0.10), replace=False))
t_obs_vdp   = t_vdp[obs_idx_vdp]
x_obs_vdp   = states_vdp[obs_idx_vdp, :1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Figure 1 — Van der Pol Oscillator Ground Truth', fontweight='bold')
axes[0].plot(t_vdp, states_vdp[:,0], color='steelblue')
axes[0].set_ylabel('x1'); axes[0].set_xlabel('Time'); axes[0].set_title('State x1')
axes[1].plot(t_vdp, states_vdp[:,1], color='tomato')
axes[1].set_ylabel('x2'); axes[1].set_xlabel('Time'); axes[1].set_title('State x2')
axes[2].plot(states_vdp[:,0], states_vdp[:,1], color='seagreen')
axes[2].set_xlabel('x1'); axes[2].set_ylabel('x2'); axes[2].set_title('Phase portrait')
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig1_vanderpol_truth.png', bbox_inches='tight')
plt.show()
print(f'VdP: {N_vdp} steps | {len(obs_idx_vdp)} sparse obs | Figure 1 saved.')

---
## Figure 2 — Quadrotor 6-DOF Simulation (Properly Fixed)

**Three fixes applied together:**
- Torques 10x smaller → angles stay < 0.3 rad → no tumbling
- Altitude feedback on thrust → z stays near 5m
- 5 second window → no long-horizon drift

In [ ]:
# CELL 4: Quadrotor simulation — properly fixed
m=1.0; g=9.81; Ixx=0.0121; Iyy=0.0121; Izz=0.0224
t_end=5.0; dt=0.01                          # FIX C: 5 seconds, not 10
t_eval=np.arange(0, t_end, dt); N=len(t_eval)

def quadrotor_dynamics(t, state, T, tau_phi, tau_theta, tau_psi):
    x,y,z,vx,vy,vz,phi,theta,psi,p,q,r = state
    ax = (np.cos(phi)*np.sin(theta)*np.cos(psi)+np.sin(phi)*np.sin(psi))*T/m
    ay = (np.cos(phi)*np.sin(theta)*np.sin(psi)-np.sin(phi)*np.cos(psi))*T/m
    az = np.cos(phi)*np.cos(theta)*T/m - g
    ct = np.cos(theta)+1e-6
    phi_dot   = p+(q*np.sin(phi)+r*np.cos(phi))*np.tan(theta)
    theta_dot = q*np.cos(phi)-r*np.sin(phi)
    psi_dot   = (q*np.sin(phi)+r*np.cos(phi))/ct
    p_dot = (Iyy-Izz)/Ixx*q*r + tau_phi/Ixx
    q_dot = (Izz-Ixx)/Iyy*p*r + tau_theta/Iyy
    r_dot = (Ixx-Iyy)/Izz*p*q + tau_psi/Izz
    return [vx,vy,vz,ax,ay,az,phi_dot,theta_dot,psi_dot,p_dot,q_dot,r_dot]

# FIX A: Torques 10x smaller than before — angles stay bounded
tau_phi_arr   = 0.0005*np.sin(0.5*t_eval)   # was 0.005
tau_theta_arr = 0.0005*np.cos(0.4*t_eval)   # was 0.005
tau_psi_arr   = 0.0002*np.sin(0.2*t_eval)   # was 0.002

# FIX B: Altitude feedback — thrust compensates for angle-induced altitude loss
# Pre-compute thrust with a simple PD altitude controller
z_target = 5.0
Kp_z = 2.0; Kd_z = 1.0
z_sim = z_target; vz_sim = 0.0
T_thrust = np.zeros(N)
for k in range(N):
    # Simple PD: T = mg + Kp*(z_target-z) + Kd*(0-vz)
    T_thrust[k] = m*g + Kp_z*(z_target - z_sim) + Kd_z*(0.0 - vz_sim)
    T_thrust[k] = np.clip(T_thrust[k], 0.5*m*g, 1.5*m*g)  # safety clip
    # Simple Euler to estimate next z for feedforward
    az_est = np.cos(0.02)*np.cos(0.02)*T_thrust[k]/m - g   # small angle approx
    vz_sim += az_est * dt
    z_sim  += vz_sim * dt

x0 = [0, 0, 5, 0, 0, 0, 0.01, 0.01, 0, 0, 0, 0]  # small initial angles

sol = solve_ivp(
    lambda t,s: quadrotor_dynamics(t,s,
        np.interp(t,t_eval,T_thrust),
        np.interp(t,t_eval,tau_phi_arr),
        np.interp(t,t_eval,tau_theta_arr),
        np.interp(t,t_eval,tau_psi_arr)),
    (0,t_end), x0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-10)
states_true = sol.y.T
state_names = ['x','y','z','vx','vy','vz','phi','theta','psi','p','q','r']

print('State ranges after all fixes:')
for i,name in enumerate(state_names):
    print(f'  {name:>6s}: [{states_true[:,i].min():.4f}, {states_true[:,i].max():.4f}]')

# Check z stays bounded
z_min = states_true[:,2].min(); z_max = states_true[:,2].max()
print(f'\nAltitude z range: [{z_min:.2f}, {z_max:.2f}] m  (should be near 5m)')
phi_max = np.abs(states_true[:,6]).max()
print(f'Max |phi|: {phi_max:.4f} rad  (should be < 0.3 rad)')

# Sparse observations: x,y,z at 10% of points
obs_indices = np.sort(np.random.choice(N, int(N*0.10), replace=False))
t_obs = t_eval[obs_indices]
x_obs = states_true[obs_indices, :3]
obs_set = set(obs_indices)

fig,axes=plt.subplots(4,3,figsize=(14,10))
fig.suptitle('Figure 2 — Quadrotor True State Trajectories (Fixed)',fontweight='bold')
for i,ax in enumerate(axes.flatten()):
    ax.plot(t_eval,states_true[:,i],color='steelblue')
    ax.set_ylabel(state_names[i]); ax.set_xlabel('Time [s]'); ax.set_title(state_names[i])
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig2_quadrotor_truth.png',bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

---
## PINN Class, Normalization & Physics Residuals

In [ ]:
# CELL 5: PINN class & normalization — FIX D: normalize_obs everywhere
class PINN(nn.Module):
    def __init__(self, out_dim=12, hidden_dim=128, n_layers=6):
        super().__init__()
        layers=[nn.Linear(1,hidden_dim), nn.Tanh()]
        for _ in range(n_layers-1):
            layers+=[nn.Linear(hidden_dim,hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim,out_dim))
        self.net=nn.Sequential(*layers)
        for mod in self.net:
            if isinstance(mod,nn.Linear):
                nn.init.xavier_normal_(mod.weight)
                nn.init.zeros_(mod.bias)
    def forward(self,t): return self.net(t)

# Normalization — fit on full 12-state ground truth
state_min   = states_true.min(axis=0)         # shape (12,)
state_max   = states_true.max(axis=0)         # shape (12,)
state_range = state_max - state_min + 1e-8    # shape (12,)

def normalize(s):     return 2*(s - state_min)/state_range - 1          # full 12-state
def denormalize(s):   return (s + 1)/2 * state_range + state_min        # full 12-state
def normalize_obs(s): return 2*(s - state_min[:3])/state_range[:3] - 1 # 3-state obs only

# Tensors
t_all_t   = torch.tensor(t_eval/t_end, dtype=torch.float32).unsqueeze(1).to(device)
t_obs_t   = torch.tensor(t_obs/t_end,  dtype=torch.float32).unsqueeze(1).to(device)
x_obs_t   = torch.tensor(normalize_obs(x_obs), dtype=torch.float32).to(device)  # (M,3)
x_ic_t    = torch.tensor(normalize(np.array(x0)), dtype=torch.float32).to(device)  # (12,)
T_t       = torch.tensor(T_thrust,      dtype=torch.float32).to(device)
tph_t     = torch.tensor(tau_phi_arr,   dtype=torch.float32).to(device)
tth_t     = torch.tensor(tau_theta_arr, dtype=torch.float32).to(device)
tps_t     = torch.tensor(tau_psi_arr,   dtype=torch.float32).to(device)
sr_t      = torch.tensor(state_range,   dtype=torch.float32).to(device)
sm_t      = torch.tensor(state_min,     dtype=torch.float32).to(device)

ci       = np.linspace(0, N-1, 500, dtype=int)
t_col    = t_all_t[ci]; T_col=T_t[ci]; tph_col=tph_t[ci]; tth_col=tth_t[ci]; tps_col=tps_t[ci]

print('Normalization ready.')
print(f'  x_obs_t shape : {x_obs_t.shape}')   # should be (50, 3)
print(f'  x_ic_t shape  : {x_ic_t.shape}')    # should be (12,)
print(f'  State ranges normalized (should all be [-1,1]):')
states_norm = normalize(states_true)
for i,name in enumerate(state_names):
    print(f'    {name:>6s}: [{states_norm[:,i].min():.2f}, {states_norm[:,i].max():.2f}]')

In [ ]:
# CELL 6: Physics residuals
def quad_physics_residual(model, t_tensor, T_arr, tph, tth, tps):
    t_r = t_tensor.clone().requires_grad_(True)
    xn  = model(t_r)                          # normalized output
    xp  = (xn+1)/2*sr_t + sm_t               # denormalized physical states
    dxn_dt = torch.zeros_like(xn)
    for i in range(12):
        grad = torch.autograd.grad(xn[:,i].sum(), t_r, create_graph=True)[0]
        dxn_dt[:,i] = grad.squeeze()/t_end
    dxdt = dxn_dt*(sr_t/2)                   # chain rule: physical derivative
    vx_,vy_,vz_      = xp[:,3],xp[:,4],xp[:,5]
    phi_,theta_,psi_ = xp[:,6],xp[:,7],xp[:,8]
    p_,q_,r_         = xp[:,9],xp[:,10],xp[:,11]
    ax_=(torch.cos(phi_)*torch.sin(theta_)*torch.cos(psi_)+torch.sin(phi_)*torch.sin(psi_))*T_arr/m
    ay_=(torch.cos(phi_)*torch.sin(theta_)*torch.sin(psi_)-torch.sin(phi_)*torch.cos(psi_))*T_arr/m
    az_=torch.cos(phi_)*torch.cos(theta_)*T_arr/m - g
    ct =torch.cos(theta_)+1e-6
    phd=p_+(q_*torch.sin(phi_)+r_*torch.cos(phi_))*torch.tan(theta_)
    thd=q_*torch.cos(phi_)-r_*torch.sin(phi_)
    psd=(q_*torch.sin(phi_)+r_*torch.cos(phi_))/ct
    pd =(Iyy-Izz)/Ixx*q_*r_+tph/Ixx
    qd =(Izz-Ixx)/Iyy*p_*r_+tth/Iyy
    rd =(Ixx-Iyy)/Izz*p_*q_+tps/Izz
    f  = torch.stack([vx_,vy_,vz_,ax_,ay_,az_,phd,thd,psd,pd,qd,rd],dim=1)
    return (dxdt-f)/(sr_t/2+1e-8)           # normalized residual

def vdp_physics_residual(model, t_tensor):
    t_r=t_tensor.clone().requires_grad_(True)
    xp=model(t_r)
    dxdt=torch.zeros_like(xp)
    for i in range(2):
        grad=torch.autograd.grad(xp[:,i].sum(),t_r,create_graph=True)[0]
        dxdt[:,i]=grad.squeeze()/t_end_vdp
    x1_,x2_=xp[:,0],xp[:,1]
    f=torch.stack([x2_, mu_vdp*(1-x1_**2)*x2_-x1_],dim=1)
    return dxdt-f

print('Physics residual functions defined.')

---
## Figure 3 — PINN Training

In [ ]:
# CELL 7: Train VdP PINN
tn_vdp      = t_vdp/t_end_vdp
t_all_vdp   = torch.tensor(tn_vdp, dtype=torch.float32).unsqueeze(1).to(device)
t_obs_vdp_t = torch.tensor(t_obs_vdp/t_end_vdp, dtype=torch.float32).unsqueeze(1).to(device)
x_obs_vdp_t = torch.tensor(x_obs_vdp, dtype=torch.float32).to(device)
x_ic_vdp_t  = torch.tensor(x0_vdp, dtype=torch.float32).to(device)
ci_vdp      = np.linspace(0, N_vdp-1, 500, dtype=int)
t_col_vdp   = t_all_vdp[ci_vdp]

model_vdp = PINN(out_dim=2, hidden_dim=64, n_layers=4).to(device)
opt_vdp   = optim.Adam(model_vdp.parameters(), lr=5e-4)
sch_vdp   = optim.lr_scheduler.CosineAnnealingLR(opt_vdp, T_max=5000, eta_min=1e-5)
hist_vdp  = {'total':[], 'data':[], 'phys':[], 'ic':[]}

print('Training PINN — Van der Pol (5000 epochs):')
t0=time.time()
for ep in tqdm(range(5000), desc='VdP PINN', ncols=70):
    opt_vdp.zero_grad()
    l_d  = nn.MSELoss()(model_vdp(t_obs_vdp_t)[:,:1], x_obs_vdp_t)
    res  = vdp_physics_residual(model_vdp, t_col_vdp)
    l_p  = (res**2).mean()
    l_ic = nn.MSELoss()(model_vdp(torch.zeros(1,1,device=device)).squeeze(), x_ic_vdp_t)
    loss = l_d + 0.1*l_p + 10.0*l_ic
    loss.backward(); opt_vdp.step(); sch_vdp.step()
    hist_vdp['total'].append(loss.item()); hist_vdp['data'].append(l_d.item())
    hist_vdp['phys'].append(l_p.item());  hist_vdp['ic'].append(l_ic.item())
tt_vdp=time.time()-t0
print(f'Done. Time: {tt_vdp:.1f}s | Final loss: {hist_vdp["total"][-1]:.4e}')

In [ ]:
# CELL 8: Train Quadrotor PINN
model_quad = PINN(out_dim=12, hidden_dim=128, n_layers=6).to(device)
opt_quad   = optim.Adam(model_quad.parameters(), lr=5e-4)
sch_quad   = optim.lr_scheduler.CosineAnnealingLR(opt_quad, T_max=10000, eta_min=1e-5)
hist_quad  = {'total':[], 'data':[], 'phys':[], 'ic':[]}

print('Training PINN — Quadrotor (10000 epochs):')
t0=time.time()
for ep in tqdm(range(10000), desc='Quad PINN', ncols=70):
    opt_quad.zero_grad()
    l_d  = nn.MSELoss()(model_quad(t_obs_t)[:,:3], x_obs_t)
    res  = quad_physics_residual(model_quad, t_col, T_col, tph_col, tth_col, tps_col)
    l_p  = (res**2).mean()
    l_ic = nn.MSELoss()(model_quad(torch.zeros(1,1,device=device)).squeeze(), x_ic_t)
    loss = l_d + 0.05*l_p + 20.0*l_ic
    loss.backward(); opt_quad.step(); sch_quad.step()
    hist_quad['total'].append(loss.item()); hist_quad['data'].append(l_d.item())
    hist_quad['phys'].append(l_p.item());  hist_quad['ic'].append(l_ic.item())
tt_quad=time.time()-t0
print(f'Done. Time: {tt_quad:.1f}s | Final loss: {hist_quad["total"][-1]:.4e}')

In [ ]:
# CELL 9: Figure 3 — Training loss curves
fig,axes=plt.subplots(2,2,figsize=(13,8))
fig.suptitle('Figure 3 — PINN Training Loss Curves',fontweight='bold')
for row,hist,label in zip([0,1],[hist_vdp,hist_quad],['Van der Pol','Quadrotor']):
    axes[row,0].semilogy(hist['total'],'k-',label='Total')
    axes[row,0].semilogy(hist['data'],'b--',label='Data')
    axes[row,0].semilogy(hist['phys'],'r-.',label='Physics')
    axes[row,0].semilogy(hist['ic'],'g:',label='IC')
    axes[row,0].set_title(f'{label} — All losses'); axes[row,0].legend(); axes[row,0].set_xlabel('Epoch')
    axes[row,1].semilogy(hist['data'],'b-')
    axes[row,1].set_title(f'{label} — Data loss only'); axes[row,1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig3_training_loss.png',bbox_inches='tight')
plt.show(); print('Figure 3 saved.')

---
## Figure 4 — PINN Estimation vs Ground Truth

In [ ]:
# CELL 10: Predictions
model_vdp.eval(); model_quad.eval()
with torch.no_grad():
    x_pinn_vdp  = model_vdp(t_all_vdp).cpu().numpy()
    x_pinn_quad = denormalize(model_quad(t_all_t).cpu().numpy())

rmse_pinn_vdp  = np.sqrt(np.mean((x_pinn_vdp-states_vdp)**2, axis=0))
rmse_pinn_quad = np.sqrt(np.mean((x_pinn_quad-states_true)**2, axis=0))
print(f'VdP  RMSE: x1={rmse_pinn_vdp[0]:.4f}, x2={rmse_pinn_vdp[1]:.4f}')
print(f'Quad mean RMSE: {np.mean(rmse_pinn_quad):.4f}')
for i,n in enumerate(state_names): print(f'  {n:>6s}: {rmse_pinn_quad[i]:.4f}')

# VdP plot
fig,axes=plt.subplots(1,2,figsize=(12,4))
fig.suptitle('Figure 4a — Van der Pol PINN Estimation',fontweight='bold')
for i,ax in enumerate(axes):
    ax.plot(t_vdp,states_vdp[:,i],'k-',label='True',lw=1.5)
    ax.plot(t_vdp,x_pinn_vdp[:,i],'r--',label='PINN',lw=1.5)
    if i==0: ax.scatter(t_obs_vdp,x_obs_vdp[:,0],s=8,c='steelblue',zorder=5,label='Obs.')
    ax.set_title(f'VdP x{i+1} | RMSE={rmse_pinn_vdp[i]:.4f}')
    ax.set_xlabel('Time'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR + 'fig4a_vdp_estimation.png',bbox_inches='tight'); plt.show()

# Quadrotor plot
fig,axes=plt.subplots(4,3,figsize=(14,10))
fig.suptitle('Figure 4b — Quadrotor PINN State Estimation vs Ground Truth',fontweight='bold')
for i,ax in enumerate(axes.flatten()):
    ax.plot(t_eval,states_true[:,i],'k-',label='True',lw=1.5,alpha=0.8)
    ax.plot(t_eval,x_pinn_quad[:,i],'r--',label='PINN',lw=1.5)
    if i<3: ax.scatter(t_obs,x_obs[:,i],s=8,c='steelblue',zorder=5,label='Obs.')
    ax.set_ylabel(state_names[i]); ax.set_xlabel('Time [s]')
    ax.set_title(f'{state_names[i]} | RMSE={rmse_pinn_quad[i]:.4f}')
    if i==0: ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig4_pinn_estimation.png',bbox_inches='tight')
plt.show(); print('Figure 4 saved.')

---
## Figures 5 & 6 — EKF, UKF, Particle Filter Baselines

In [ ]:
# CELL 11: EKF
H=np.zeros((3,12)); H[0,0]=1; H[1,1]=1; H[2,2]=1
Q_ekf=np.diag([1e-4]*3+[1e-3]*3+[1e-4]*3+[1e-3]*3)
R_ekf=np.diag([0.01]*3)
meas=states_true[:,:3]+np.random.randn(N,3)*0.05

def linearize_F(state, dt_):
    eps=1e-5; f0=np.array(quadrotor_dynamics(0,state,m*g,0,0,0))
    J=np.zeros((12,12))
    for j in range(12):
        sp=state.copy(); sp[j]+=eps
        J[:,j]=(np.array(quadrotor_dynamics(0,sp,m*g,0,0,0))-f0)/eps
    return np.eye(12)+J*dt_

xe=np.array(x0)+np.random.randn(12)*0.02; Pe=np.eye(12)*0.1
x_ekf=np.zeros((N,12)); x_ekf[0]=xe
t0_ekf=time.time()
for k in tqdm(range(1,N),desc='EKF',ncols=50,leave=False):
    dyn=np.array(quadrotor_dynamics(t_eval[k],xe,T_thrust[k],tau_phi_arr[k],tau_theta_arr[k],tau_psi_arr[k]))
    xp=xe+dyn*dt; Fj=linearize_F(xe,dt); Pp=Fj@Pe@Fj.T+Q_ekf
    if k in obs_set:
        S=H@Pp@H.T+R_ekf; K=Pp@H.T@np.linalg.inv(S)
        xe=xp+K@(meas[k]-H@xp); Pe=(np.eye(12)-K@H)@Pp
    else: xe=xp; Pe=Pp
    x_ekf[k]=xe
time_ekf=time.time()-t0_ekf
rmse_ekf=np.sqrt(np.mean((x_ekf-states_true)**2,axis=0))
print(f'EKF: Time={time_ekf:.2f}s | Mean RMSE={np.mean(rmse_ekf):.4f}')

In [ ]:
# CELL 12: UKF
n_s=12; alpha=1e-3; kappa=0.0; beta=2.0; lam=alpha**2*(n_s+kappa)-n_s
Wm=np.full(2*n_s+1,1/(2*(n_s+lam))); Wm[0]=lam/(n_s+lam)
Wc=Wm.copy(); Wc[0]+=1-alpha**2+beta

def sigma_pts(mu,P):
    S=np.linalg.cholesky((n_s+lam)*P+np.eye(n_s)*1e-8)
    pts=np.zeros((2*n_s+1,n_s)); pts[0]=mu
    for i in range(n_s): pts[i+1]=mu+S[:,i]; pts[n_s+i+1]=mu-S[:,i]
    return pts

def quad_f(s,k):
    return s+np.array(quadrotor_dynamics(t_eval[k],s,T_thrust[k],tau_phi_arr[k],tau_theta_arr[k],tau_psi_arr[k]))*dt

xu=np.array(x0)+np.random.randn(12)*0.02; Pu=np.eye(12)*0.1
x_ukf=np.zeros((N,12)); x_ukf[0]=xu
t0_ukf=time.time()
for k in tqdm(range(1,N),desc='UKF',ncols=50,leave=False):
    sp=sigma_pts(xu,Pu)
    sp_f=np.array([quad_f(sp[i],k) for i in range(2*n_s+1)])
    xu_p=np.sum(Wm[:,None]*sp_f,axis=0)
    Pu_p=sum(Wc[i]*np.outer(sp_f[i]-xu_p,sp_f[i]-xu_p) for i in range(2*n_s+1))+Q_ekf
    if k in obs_set:
        sp2=sigma_pts(xu_p,Pu_p)
        zp=np.array([H@sp2[i] for i in range(2*n_s+1)])
        zm=np.sum(Wm[:,None]*zp,axis=0)
        Sz=sum(Wc[i]*np.outer(zp[i]-zm,zp[i]-zm) for i in range(2*n_s+1))+R_ekf
        Pxz=sum(Wc[i]*np.outer(sp2[i]-xu_p,zp[i]-zm) for i in range(2*n_s+1))
        Ku=Pxz@np.linalg.inv(Sz)
        xu=xu_p+Ku@(meas[k]-zm); Pu=Pu_p-Ku@Sz@Ku.T
    else: xu=xu_p; Pu=Pu_p
    x_ukf[k]=xu
time_ukf=time.time()-t0_ukf
rmse_ukf=np.sqrt(np.mean((x_ukf-states_true)**2,axis=0))
print(f'UKF: Time={time_ukf:.2f}s | Mean RMSE={np.mean(rmse_ukf):.4f}')

In [ ]:
# CELL 13: Particle Filter
n_p=300
particles=np.array(x0)+np.random.randn(n_p,12)*0.05
weights=np.ones(n_p)/n_p
x_pf=np.zeros((N,12)); x_pf[0]=np.average(particles,weights=weights,axis=0)
t0_pf=time.time()
for k in tqdm(range(1,N),desc='PF',ncols=50,leave=False):
    for i in range(n_p):
        dyn=np.array(quadrotor_dynamics(t_eval[k],particles[i],
            T_thrust[k],tau_phi_arr[k],tau_theta_arr[k],tau_psi_arr[k]))
        particles[i]=particles[i]+dyn*dt+np.random.randn(12)*0.01
    if k in obs_set:
        for i in range(n_p):
            err=meas[k]-H@particles[i]
            weights[i]*=np.exp(-0.5*err@np.linalg.inv(R_ekf)@err+1e-300)
        weights+=1e-300; weights/=weights.sum()
        idx=np.random.choice(n_p,n_p,p=weights)
        particles=particles[idx]; weights=np.ones(n_p)/n_p
    x_pf[k]=np.average(particles,weights=weights,axis=0)
time_pf=time.time()-t0_pf
rmse_pf=np.sqrt(np.mean((x_pf-states_true)**2,axis=0))
print(f'PF: Time={time_pf:.2f}s | Mean RMSE={np.mean(rmse_pf):.4f}')

In [ ]:
# CELL 14: Figure 5 — Full comparison
methods=['PINN\n(ours)','EKF','UKF','PF']
colors=['tomato','steelblue','seagreen','darkorange']
means=[np.mean(rmse_pinn_quad),np.mean(rmse_ekf),np.mean(rmse_ukf),np.mean(rmse_pf)]

fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Figure 5 — PINN vs EKF vs UKF vs Particle Filter',fontweight='bold')
x_pos=np.arange(12); w=0.2
axes[0].bar(x_pos-1.5*w,rmse_pinn_quad,w,label='PINN',color='tomato',alpha=0.85)
axes[0].bar(x_pos-0.5*w,rmse_ekf,w,label='EKF',color='steelblue',alpha=0.85)
axes[0].bar(x_pos+0.5*w,rmse_ukf,w,label='UKF',color='seagreen',alpha=0.85)
axes[0].bar(x_pos+1.5*w,rmse_pf,w,label='PF',color='darkorange',alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(state_names,fontsize=9)
axes[0].set_ylabel('RMSE'); axes[0].set_title('Per-state RMSE'); axes[0].legend()
bars=axes[1].bar(methods,means,color=colors,alpha=0.85,width=0.5)
axes[1].set_ylabel('Mean RMSE'); axes[1].set_title('Overall mean RMSE')
for bar,v in zip(bars,means):
    axes[1].text(bar.get_x()+bar.get_width()/2,v*1.02,f'{v:.4f}',ha='center',fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig5_all_methods_comparison.png',bbox_inches='tight')
plt.show(); print('Figure 5 saved.')

In [ ]:
# CELL 15: Figure 6 — Trajectory comparison
fig,axes=plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Figure 6 — State Trajectory Comparison',fontweight='bold')
for ax,si in zip(axes,[2,5,6]):
    ax.plot(t_eval,states_true[:,si],'k-',label='Truth',lw=2)
    ax.plot(t_eval,x_pinn_quad[:,si],'r--',label='PINN',lw=1.5)
    ax.plot(t_eval,x_ekf[:,si],'b:',label='EKF',lw=1.5)
    ax.plot(t_eval,x_ukf[:,si],'g-.',label='UKF',lw=1.5)
    ax.plot(t_eval,x_pf[:,si],'m--',label='PF',lw=1,alpha=0.7)
    ax.set_xlabel('Time [s]'); ax.set_ylabel(state_names[si])
    ax.set_title(f'State: {state_names[si]}'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig6_trajectory_comparison.png',bbox_inches='tight')
plt.show(); print('Figure 6 saved.')

---
## Figure 7 — Ablation Study

In [ ]:
# CELL 16: Pure NN (no physics)
model_nn=PINN(out_dim=12,hidden_dim=128,n_layers=6).to(device)
opt_nn=optim.Adam(model_nn.parameters(),lr=5e-4)
sch_nn=optim.lr_scheduler.CosineAnnealingLR(opt_nn,T_max=10000,eta_min=1e-5)
print('Training pure NN (no physics) — 10000 epochs:')
for ep in tqdm(range(10000),desc='Pure NN',ncols=70):
    opt_nn.zero_grad()
    l=nn.MSELoss()(model_nn(t_obs_t)[:,:3], x_obs_t)
    l.backward(); opt_nn.step(); sch_nn.step()
model_nn.eval()
with torch.no_grad():
    x_nn=denormalize(model_nn(t_all_t).cpu().numpy())
rmse_nn=np.sqrt(np.mean((x_nn-states_true)**2,axis=0))
print(f'Pure NN RMSE : {np.mean(rmse_nn):.4f}')
print(f'PINN    RMSE : {np.mean(rmse_pinn_quad):.4f}')
print(f'Improvement  : {(np.mean(rmse_nn)-np.mean(rmse_pinn_quad))/np.mean(rmse_nn)*100:.1f}%')

fig,axes=plt.subplots(1,2,figsize=(12,5))
fig.suptitle('Figure 7 — Ablation: Physics Term Contribution',fontweight='bold')
x_pos=np.arange(12); w=0.35
axes[0].bar(x_pos-w/2,rmse_pinn_quad,w,label='PINN (physics)',color='tomato',alpha=0.85)
axes[0].bar(x_pos+w/2,rmse_nn,w,label='NN (no physics)',color='gray',alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(state_names,fontsize=9)
axes[0].set_ylabel('RMSE'); axes[0].set_title('Per-state RMSE'); axes[0].legend()
mp=np.mean(rmse_pinn_quad); mn_=np.mean(rmse_nn)
bars2=axes[1].bar(['PINN\n(physics)','NN\n(no physics)'],[mp,mn_],
                  color=['tomato','gray'],alpha=0.85,width=0.4)
axes[1].set_ylabel('Mean RMSE'); axes[1].set_title('Overall comparison')
for bar,v in zip(bars2,[mp,mn_]):
    axes[1].text(bar.get_x()+bar.get_width()/2,v*1.02,f'{v:.4f}',ha='center',fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig7_ablation.png',bbox_inches='tight')
plt.show(); print('Figure 7 saved.')

---
## Figure 8 — Sparsity Sweep

In [ ]:
# CELL 17: Sparsity sweep — normalize_obs used throughout
sparsity_levels=[0.02,0.05,0.10,0.20,0.35,0.50]
rmse_sp_pinn=[]; rmse_sp_nn=[]; rmse_sp_ekf=[]

for sp in sparsity_levels:
    idx_sp=np.sort(np.random.choice(N,int(N*sp),replace=False))
    t_sp=torch.tensor(t_eval[idx_sp]/t_end,dtype=torch.float32).unsqueeze(1).to(device)
    x_sp=torch.tensor(normalize_obs(states_true[idx_sp,:3]),dtype=torch.float32).to(device)  # FIX D

    # PINN
    mp=PINN(out_dim=12,hidden_dim=128,n_layers=6).to(device)
    op=optim.Adam(mp.parameters(),lr=5e-4)
    sc=optim.lr_scheduler.CosineAnnealingLR(op,T_max=5000,eta_min=1e-5)
    x_ic_sp=torch.tensor(normalize(np.array(x0)),dtype=torch.float32).to(device)
    for ep in tqdm(range(5000),desc=f'sp={sp:.0%} PINN',ncols=60,leave=False):
        op.zero_grad()
        l_d=nn.MSELoss()(mp(t_sp)[:,:3], x_sp)
        res=quad_physics_residual(mp,t_col,T_col,tph_col,tth_col,tps_col)
        l_p=(res**2).mean()
        l_ic=nn.MSELoss()(mp(torch.zeros(1,1,device=device)).squeeze(), x_ic_sp)
        (l_d+0.05*l_p+20*l_ic).backward(); op.step(); sc.step()
    mp.eval()
    with torch.no_grad(): xp_=denormalize(mp(t_all_t).cpu().numpy())
    rmse_sp_pinn.append(np.sqrt(np.mean((xp_-states_true)**2)))

    # Pure NN
    mn=PINN(out_dim=12,hidden_dim=128,n_layers=6).to(device)
    on=optim.Adam(mn.parameters(),lr=5e-4)
    scn=optim.lr_scheduler.CosineAnnealingLR(on,T_max=5000,eta_min=1e-5)
    for ep in tqdm(range(5000),desc=f'sp={sp:.0%} NN',ncols=60,leave=False):
        on.zero_grad()
        l=nn.MSELoss()(mn(t_sp)[:,:3], x_sp)
        l.backward(); on.step(); scn.step()
    mn.eval()
    with torch.no_grad(): xn_=denormalize(mn(t_all_t).cpu().numpy())
    rmse_sp_nn.append(np.sqrt(np.mean((xn_-states_true)**2)))

    # EKF
    obs_sp=set(idx_sp)
    xe_sp=np.array(x0)+np.random.randn(12)*0.02; Pe_sp=np.eye(12)*0.1
    xe_all=np.zeros((N,12)); xe_all[0]=xe_sp
    for k in range(1,N):
        dyn=np.array(quadrotor_dynamics(t_eval[k],xe_sp,T_thrust[k],tau_phi_arr[k],tau_theta_arr[k],tau_psi_arr[k]))
        xep=xe_sp+dyn*dt; Fj=linearize_F(xe_sp,dt); Pp=Fj@Pe_sp@Fj.T+Q_ekf
        if k in obs_sp:
            S=H@Pp@H.T+R_ekf; K=Pp@H.T@np.linalg.inv(S)
            xe_sp=xep+K@(meas[k]-H@xep); Pe_sp=(np.eye(12)-K@H)@Pp
        else: xe_sp=xep; Pe_sp=Pp
        xe_all[k]=xe_sp
    rmse_sp_ekf.append(np.sqrt(np.mean((xe_all-states_true)**2)))
    print(f'sp={sp:.0%} → PINN={rmse_sp_pinn[-1]:.4f} | NN={rmse_sp_nn[-1]:.4f} | EKF={rmse_sp_ekf[-1]:.4f}')

fig,ax=plt.subplots(figsize=(8,5))
ax.plot([s*100 for s in sparsity_levels],rmse_sp_pinn,'r-o',label='PINN (ours)',lw=2)
ax.plot([s*100 for s in sparsity_levels],rmse_sp_nn,'k-^',label='NN (no physics)',lw=2)
ax.plot([s*100 for s in sparsity_levels],rmse_sp_ekf,'b-s',label='EKF',lw=2)
ax.set_xlabel('Observation density [%]'); ax.set_ylabel('Mean RMSE')
ax.set_title('Figure 8 — Estimation Error vs Observation Sparsity',fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig8_sparsity_sweep.png',bbox_inches='tight')
plt.show(); print('Figure 8 saved.')

---
## Figure 9 — Robustness Analysis

In [ ]:
# CELL 18: Noise robustness — normalize_obs used
noise_levels=[0.01,0.05,0.10,0.20,0.35,0.50]
rmse_n_pinn=[]; rmse_n_ekf=[]

for sigma in noise_levels:
    xo_n = normalize_obs(x_obs) + np.random.randn(*x_obs.shape)*sigma  # FIX D
    xo_n_t=torch.tensor(xo_n,dtype=torch.float32).to(device)
    mn=PINN(out_dim=12,hidden_dim=128,n_layers=6).to(device)
    on=optim.Adam(mn.parameters(),lr=5e-4)
    scn=optim.lr_scheduler.CosineAnnealingLR(on,T_max=4000,eta_min=1e-5)
    for ep in tqdm(range(4000),desc=f'sigma={sigma}',ncols=60,leave=False):
        on.zero_grad()
        l_d=nn.MSELoss()(mn(t_obs_t)[:,:3], xo_n_t)
        res=quad_physics_residual(mn,t_col,T_col,tph_col,tth_col,tps_col)
        l_p=(res**2).mean()
        l_ic=nn.MSELoss()(mn(torch.zeros(1,1,device=device)).squeeze(), x_ic_t)
        (l_d+0.05*l_p+20*l_ic).backward(); on.step(); scn.step()
    mn.eval()
    with torch.no_grad(): xpn=denormalize(mn(t_all_t).cpu().numpy())
    rmse_n_pinn.append(np.sqrt(np.mean((xpn-states_true)**2)))

    Rn=np.diag([sigma**2]*3)
    xe_n=np.array(x0)+np.random.randn(12)*0.02; Pe_n=np.eye(12)*0.1
    xen_all=np.zeros((N,12)); xen_all[0]=xe_n
    mn_obs=states_true[:,:3]+np.random.randn(N,3)*sigma
    for k in range(1,N):
        dyn=np.array(quadrotor_dynamics(t_eval[k],xe_n,T_thrust[k],tau_phi_arr[k],tau_theta_arr[k],tau_psi_arr[k]))
        xep=xe_n+dyn*dt; Fj=linearize_F(xe_n,dt); Pp=Fj@Pe_n@Fj.T+Q_ekf
        if k in obs_set:
            S=H@Pp@H.T+Rn; K=Pp@H.T@np.linalg.inv(S)
            xe_n=xep+K@(mn_obs[k]-H@xep); Pe_n=(np.eye(12)-K@H)@Pp
        else: xe_n=xep; Pe_n=Pp
        xen_all[k]=xe_n
    rmse_n_ekf.append(np.sqrt(np.mean((xen_all-states_true)**2)))
    print(f'sigma={sigma:.2f} → PINN={rmse_n_pinn[-1]:.4f} | EKF={rmse_n_ekf[-1]:.4f}')

In [ ]:
# CELL 19: Parameter uncertainty
perturbations=[0.0,0.05,0.10,0.20,0.30]
rmse_p_pinn=[]; rmse_p_ekf=[]

for perturb in perturbations:
    mp_=m*(1+perturb*np.random.randn())
    sol_p=solve_ivp(
        lambda t,s: quadrotor_dynamics(t,s,
            np.interp(t,t_eval,T_thrust*mp_/m),
            np.interp(t,t_eval,tau_phi_arr),
            np.interp(t,t_eval,tau_theta_arr),
            np.interp(t,t_eval,tau_psi_arr)),
        (0,t_end),x0,t_eval=t_eval,method='RK45',rtol=1e-6,atol=1e-9)
    sp_=sol_p.y.T
    # Normalize for this perturbed system
    sm_p=sp_.min(axis=0); sr_p=sp_.max(axis=0)-sm_p+1e-8
    xop=2*(sp_[obs_indices,:3]-sm_p[:3])/sr_p[:3]-1
    xop_t=torch.tensor(xop,dtype=torch.float32).to(device)
    x_ic_p=torch.tensor(2*(np.array(x0)-sm_p)/sr_p-1,dtype=torch.float32).to(device)
    sr_p_t=torch.tensor(sr_p,dtype=torch.float32).to(device)
    sm_p_t=torch.tensor(sm_p,dtype=torch.float32).to(device)

    mpp=PINN(out_dim=12,hidden_dim=128,n_layers=6).to(device)
    opp=optim.Adam(mpp.parameters(),lr=5e-4)
    scp=optim.lr_scheduler.CosineAnnealingLR(opp,T_max=4000,eta_min=1e-5)
    for ep in tqdm(range(4000),desc=f'p={perturb:.0%}',ncols=60,leave=False):
        opp.zero_grad()
        l_d=nn.MSELoss()(mpp(t_obs_t)[:,:3], xop_t)
        res=quad_physics_residual(mpp,t_col,T_col,tph_col,tth_col,tps_col)
        l_p=(res**2).mean()
        l_ic=nn.MSELoss()(mpp(torch.zeros(1,1,device=device)).squeeze(), x_ic_p)
        (l_d+0.05*l_p+20*l_ic).backward(); opp.step(); scp.step()
    mpp.eval()
    with torch.no_grad(): xpp_n=mpp(t_all_t).cpu().numpy()
    xpp_=(xpp_n+1)/2*sr_p+sm_p
    rmse_p_pinn.append(np.sqrt(np.mean((xpp_-sp_)**2)))
    rmse_p_ekf.append(np.sqrt(np.mean((x_ekf-sp_)**2)))
    print(f'p={perturb:.0%} → PINN={rmse_p_pinn[-1]:.4f} | EKF={rmse_p_ekf[-1]:.4f}')

fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Figure 9 — Robustness Analysis',fontweight='bold')
axes[0].plot(noise_levels,rmse_n_pinn,'r-o',label='PINN',lw=2)
axes[0].plot(noise_levels,rmse_n_ekf,'b-s',label='EKF',lw=2)
axes[0].set_xlabel('Measurement noise sigma'); axes[0].set_ylabel('Mean RMSE')
axes[0].set_title('9A — Noise robustness'); axes[0].legend()
axes[1].plot([p*100 for p in perturbations],rmse_p_pinn,'r-o',label='PINN',lw=2)
axes[1].plot([p*100 for p in perturbations],rmse_p_ekf,'b-s',label='EKF',lw=2)
axes[1].set_xlabel('Parameter perturbation [%]'); axes[1].set_ylabel('Mean RMSE')
axes[1].set_title('9B — Model uncertainty robustness'); axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig9_robustness.png',bbox_inches='tight')
plt.show(); print('Figure 9 saved.')

---
## Figure 10 — Computational Cost

In [ ]:
# CELL 20: Compute cost
t0=time.time()
for _ in range(200):
    with torch.no_grad(): _=model_quad(t_all_t)
time_pinn_infer=(time.time()-t0)/200

fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Figure 10 — Computational Cost Comparison',fontweight='bold')
train_labels=['PINN\n(train)','EKF\n(total)','UKF\n(total)','PF\n(total)']
train_vals=[tt_quad,time_ekf,time_ukf,time_pf]
infer_labels=['PINN\n(infer)','EKF\n(step)','UKF\n(step)','PF\n(step)']
infer_vals=[time_pinn_infer,time_ekf/N,time_ukf/N,time_pf/N]
cols=['tomato','steelblue','seagreen','darkorange']
axes[0].bar(train_labels,train_vals,color=cols,alpha=0.85)
axes[0].set_ylabel('Time [s]'); axes[0].set_title('Total runtime'); axes[0].set_yscale('log')
axes[1].bar(infer_labels,infer_vals,color=cols,alpha=0.85)
axes[1].set_ylabel('Time [s]'); axes[1].set_title('Per-query inference time'); axes[1].set_yscale('log')
plt.tight_layout()
plt.savefig(OUT_DIR + 'fig10_compute_cost.png',bbox_inches='tight')
plt.show(); print('Figure 10 saved.')

In [ ]:
# CELL 21: Full summary
print('='*65)
print('FULL RESULTS SUMMARY — paste into paper notes')
print('='*65)
print(f"{'Method':<14} {'Mean RMSE':>10}")
print('-'*30)
for name,rmse in [('PINN (ours)',np.mean(rmse_pinn_quad)),
                   ('EKF',np.mean(rmse_ekf)),
                   ('UKF',np.mean(rmse_ukf)),
                   ('PF',np.mean(rmse_pf)),
                   ('NN (no phys)',np.mean(rmse_nn))]:
    tag=' <-- ours' if 'PINN' in name and 'no' not in name else ''
    print(f"{name:<14} {rmse:>10.4f}{tag}")
print()
print(f'Ablation improvement : {(np.mean(rmse_nn)-np.mean(rmse_pinn_quad))/np.mean(rmse_nn)*100:.1f}%')
print(f'VdP x1 RMSE          : {rmse_pinn_vdp[0]:.4f}')
print(f'VdP x2 RMSE          : {rmse_pinn_vdp[1]:.4f}')
print(f'Quadrotor train time : {tt_quad:.1f}s')
print(f'Inference time       : {time_pinn_infer*1000:.3f}ms per query')
print(f'Sparsity             : 10% | Observed: x,y,z only (3 of 12)')
print(f'Simulation window    : {t_end}s')
print(f'Max |phi|            : {np.abs(states_true[:,6]).max():.4f} rad')
print(f'z range              : [{states_true[:,2].min():.3f}, {states_true[:,2].max():.3f}] m')

In [ ]:
# CELL 22: Figures saved to /kaggle/working/ — list them here
import os

figs = [
    'fig1_vanderpol_truth.png',
    'fig2_quadrotor_truth.png',
    'fig3_training_loss.png',
    'fig4_pinn_estimation.png',
    'fig4a_vdp_estimation.png',
    'fig5_all_methods_comparison.png',
    'fig6_trajectory_comparison.png',
    'fig7_ablation.png',
    'fig8_sparsity_sweep.png',
    'fig9_robustness.png',
    'fig10_compute_cost.png',
]

print("Figures saved to /kaggle/working/:")
for f in figs:
    path = OUT_DIR + f
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  ✓ {f}  ({size_kb:.1f} KB)')
    else:
        print(f'  ✗ {f}  — NOT FOUND')

print("\nAll done! Download figures from the Output panel on the right.")
print("Send all figures + Cell 21 summary to Claude for paper drafting.")
